In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

'Not much—just here and ready to help. What’s going on?'

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—often you can, but it depends on the course’s enrollment rules and whether it’s still open.

A few possibilities:
- **Open enrollment:** you can join anytime.
- **Cohort-based course:** you may have to wait for the next start date.
- **In-progress class:** sometimes late entry is allowed, sometimes not.

If you want, I can help you check the course policy or draft a short message to the instructor asking if you can still enroll.


In [4]:
from openai.types.responses import file_search_tool_param
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results )
    return llm(user_prompt)

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course["path"]}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1346

In [19]:
import requests
import pprint

# 1. Fetch courses metadata
docs_url = "https://datatalks.club/faq/json/courses.json"
courses_raw = requests.get(docs_url).json()

# 2. Reset documents list
documents = []
url_prefix = "https://datatalks.club/faq"

# 3. Fetch and extract all documents
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    
    if course_response.status_code == 200:
        course_documents = course_response.json()
        documents.extend(course_documents)

# 4. Filter only machine-learning-zoomcamp
documents = [doc for doc in documents if doc['course'] == 'machine-learning-zoomcamp']

# 5. Print length and the first document to verify
print("Number of ML documents:", len(documents))
if len(documents) > 0:
    pprint.pprint(documents[0])


Number of ML documents: 472
{'answer': '- Do the tasks locally\n'
           '- Publish your code (e.g., in your own GitHub repo)\n'
           '- Submit your answers via the homework form and include the URL to '
           'your code\n'
           '- You will see the answers only after the deadline\n'
           "- Homeworks are in the cohorts folder, e.g. for 2025 it's "
           '[`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n'
           '- The forms for submitting the homework are in the [course '
           'management platform](https://courses.datatalks.club/)',
 'course': 'machine-learning-zoomcamp',
 'id': '0e38656cfb',
 'question': 'How do I submit homework?',
 'section': 'General Course-Related Questions'}


In [22]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [25]:
question = "How do I submit homework?"
index.search(query=question)


[{'id': '8351543816',
  'course': 'machine-learning-zoomcamp',
  'section': 'Module 4 Homework',
  'question': 'Homework: Why do I have different values of accuracy than the options in the homework?',
  'answer': 'One main reason behind this issue is the method of splitting the data. For example, if we want to split the data into train/validation/test with the ratios 60%/20%/20%, different methods may yield different results even if the final ratios are the same.\n\n1. Method 1:\n   \n   ```python\n   df_train, df_temp = train_test_split(df, test_size=0.4, random_state=42)\n   df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)\n   ```\n\n2. Method 2:\n   \n   ```python\n   df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)\n   df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)\n   ```\n\nWhile both methods achieve the same ratio, the data split differently, resulting in variations in accuracy. It is re

In [26]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )